<a href="https://colab.research.google.com/github/Hiruni-Pavithrani-Gunasekara/EN3150_Assignment03_Systronix/blob/main/Systronix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Simple Convolutional Neural Network to Perform Classification.**
EN3150 Assignment 03

Team Systronix - UCI RealWaste dataset

### Dataset Setup & Extraction
Automatically downloads the `realwaste.zip` file from GitHub Releases and extracts it into the local Colab workspace.

In [1]:
import os
import zipfile

dataset_url = "https://github.com/Hiruni-Pavithrani-Gunasekara/EN3150_Assignment03_Systronix/releases/download/v1.0.0/realwaste.zip"
zip_path = "/content/realwaste.zip"

if not os.path.exists(zip_path):
    print("Downloading dataset from GitHub Release...")
    !wget -O {zip_path} "{dataset_url}"

print("Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("Dataset ready!")

--2026-09-23 02:32:36--  https://github.com/Hiruni-Pavithrani-Gunasekara/EN3150_Assignment03_Systronix/releases/download/v1.0.0/realwaste.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1346976624/2b8429de-6dc2-47b3-84bf-00dd1ef9ad53?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-23T03%3A06%3A45Z&rscd=attachment%3B+filename%3Drealwaste.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-23T02%3A06%3A11Z&ske=2026-09-23T03%3A06%3A45Z&sks=b&skv=2018-11-09&sig=%2B1nvnUXB6nRLEyMF9gJM7TDpNeV%2BfQAyusOizUDekk4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MDEzNDM1NiwibmJmIjoxNzkwMTMwNzU2LCJwYX

### 1. Data Preparation

In [2]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# 1. Image Transformations
transform = transforms.Compose([
    transforms.Resize((64, 64)), # Downscale to 64x64
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load Dataset
data_dir = "/content/realwaste-main/RealWaste"
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# 3. Calculate Splits (70% Train, 15% Val, 15% Test)
total_count = len(full_dataset)
train_count = int(0.70 * total_count)
val_count = int(0.15 * total_count)
test_count = total_count - train_count - val_count

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_count, val_count, test_count], generator=generator
)

# 4. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

num_classes = len(full_dataset.classes)

print("No of Classes:", num_classes)
print("Classes:", full_dataset.classes)
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print(f"Ratios: {round(len(train_dataset)/total_count * 100)}% / "f"{round(len(val_dataset)/total_count * 100)}% / {round(len(test_dataset)/total_count * 100)}%")

No of Classes: 9
Classes: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
Train: 3326
Validation: 712
Test: 714
Ratios: 70% / 15% / 15%


## **2. Custom Arcitecture Design**

### Model A - Standard CNN

In [3]:
# Define Model A: Standard CNN

class ModelA(nn.Module):
    def __init__(self, num_classes=9):
        super(ModelA, self).__init__()

        # Feature Extractor: Interleaved Conv2d and MaxPooling2d
        self.features = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 32 x 32 x 32

            # Conv Block 2
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 64 x 16 x 16

            # Conv Block 3
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)   # Output: 128 x 8 x 8
        )

        # Classifier: Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate Model A
model_a = ModelA(num_classes=num_classes)

# Count and display trainable parameters
total_params_a = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(model_a)
print(f"\nTotal Trainable Parameters in Model A: {total_params_a:,}")

ModelA(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=8192, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=9, bias=True)
  )
)

Total Trainable Parameters in Model A: 618,185


### Model B - (Lightweight CNN)

In [4]:
# Custom Depthwise Separable Convolution Block
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(DepthwiseSeparableConv, self).__init__()
        # 1. Depthwise Conv (groups = in_channels)
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            padding=padding, groups=in_channels, bias=False
        )
        # 2. Pointwise Conv (1x1 conv)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=True)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.relu(x)
        return x

# Model B: Lightweight CNN (<100k parameters)
class ModelB(nn.Module):
    def __init__(self, num_classes=9):
        super(ModelB, self).__init__()

        self.features = nn.Sequential(
            # Block 1: 3 -> 32 channels
            DepthwiseSeparableConv(3, 32),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32 x 32 x 32

            # Block 2: 32 -> 64 channels
            DepthwiseSeparableConv(32, 64),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64 x 16 x 16

            # Block 3: 64 -> 128 channels
            DepthwiseSeparableConv(64, 128),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 128 x 8 x 8

            # Block 4: 128 -> 128 channels
            DepthwiseSeparableConv(128, 128),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 128 x 4 x 4
        )

        # Global Average Pooling reduces (128, 4, 4) -> (128, 1, 1)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

# Instantiate Model B
model_b = ModelB(num_classes=num_classes)

# Count and display trainable parameters
total_params_b = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
print(model_b)
print(f"\nTotal Trainable Parameters in Model B: {total_params_b:,}")

ModelB(
  (features): Sequential(
    (0): DepthwiseSeparableConv(
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
      (pointwise): Conv2d(3, 32, kernel_size=(1, 1), stride=(1, 1))
      (relu): ReLU()
    )
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): DepthwiseSeparableConv(
      (depthwise): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
      (pointwise): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
      (relu): ReLU()
    )
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): DepthwiseSeparableConv(
      (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
      (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
      (relu): ReLU()
    )
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6